## Silver to Gold Job

Constrói a camada Gold do IMDM (Índice de Maturidade Digital Municipal) — Etapa 3 conforme `docs/gold_dimensional_model.md` e `docs/radar_imdm.md`:

- 1. Configuração, guarda de pré-requisitos (Ajuste 1) e criação do schema `semantic`
- 2. `dim_municipio` — 5.570 municípios com porte populacional derivado
- 3. Radar IMDM — parametrização dos 64 itens pontuados (dimensões, âncoras e regras)
- 4. `dim_indicador` — 360 variáveis com significados, linhagem e regras de pontuação
- 5. `fact_resposta` — despivotização atômica (5.570 × 360 = 2.005.200 linhas)
- 6. `fact_resposta` — pontuação no nível do item (formato, escada, grupos e escalonados)
- 7. `fact_resposta` — montagem final com denominador efetivo e gravação
- 8. `fact_imdm_municipio` — scores D1–D5, IMDM global e estágio de maturidade (MMD)
- 9. Controles automatizados de qualidade (Seção 6 da especificação)

> **Pré-requisito crítico (Ajuste 1):** a `silver.munic_clean_data` deve preservar os 6 estados textuais originais da MUNIC (`Sim`, `Não`, `-`, `Não sabe informar`, `Não informou`, `Recusa`) em todas as 360 colunas de indicadores — o cast destrutivo para `boolean` do `bronze_to_silver` deve ter sido revertido. O Step 1 bloqueia a execução caso o ajuste não tenha sido aplicado.

> **Nota:** os cortes do `nivel_maturidade_mmd` usam os quartis da escala 0–100 (`<25` Incipiente, `25–50` Básico, `50–75` Intermediário, `>75` Avançado) — ajustáveis em uma única expressão.

In [0]:
%run ../environment/env_variables

In [0]:
%run ../environment/env_functions

## Step 1 - Configuração, pré-requisitos e schema semântico

- Define as tabelas de origem (silver + diagnósticos `dq_*` da Etapa 2) e as tabelas gold de destino
- Valida que as 360 colunas de indicadores estão em `StringType` com os 6 estados originais preservados (Ajuste 1) — a execução é bloqueada se o cast destrutivo ainda estiver vigente
- Cria o schema `semantic` (Ajuste 3) que receberá as views do notebook `gold_to_semantic`

In [0]:
from pyspark.sql import functions as F

# Tabelas de origem (silver + diagnósticos da Etapa 2)
silver_data_table  = f"{catalog_name}.{silver_schema}.munic_clean_data"
silver_dict_table  = f"{catalog_name}.{silver_schema}.munic_clean_dict"
dq_profile_table   = f"{catalog_name}.{gold_schema}.dq_column_profile"
dq_skiplogic_table = f"{catalog_name}.{gold_schema}.dq_skiplogic_candidates"

# Tabelas gold de destino
dim_municipio_table = f"{catalog_name}.{gold_schema}.dim_municipio"
dim_indicador_table = f"{catalog_name}.{gold_schema}.dim_indicador"
fact_resposta_table = f"{catalog_name}.{gold_schema}.fact_resposta"
fact_imdm_table     = f"{catalog_name}.{gold_schema}.fact_imdm_municipio"

# Ajuste 3: schema semântico para as views do gold_to_semantic
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.semantic")

# Carrega a edição corrente da silver
silver_df = spark.table(silver_data_table).filter(F.col("ano_edicao") == reference_year)
indicator_cols = [c for c in silver_df.columns if c.startswith(("mtic", "mgov"))]

# Guarda de pré-requisito (Ajuste 1): indicadores devem preservar os 6 estados originais
assert len(indicator_cols) == 360, (
    f"Esperadas 360 colunas de indicadores em {silver_data_table}, encontradas {len(indicator_cols)}."
)
non_string_cols = [c for c in indicator_cols if silver_df.schema[c].dataType.simpleString() != "string"]
if non_string_cols:
    raise Exception(
        "Pré-requisito 'Ajuste 1' NÃO aplicado: colunas de indicadores não são StringType "
        f"(cast destrutivo ainda vigente, ex.: {non_string_cols[:8]}...). "
        "Reexecute o bronze_to_silver preservando os 6 estados originais da MUNIC "
        "('Sim', 'Não', '-', 'Não sabe informar', 'Não informou', 'Recusa') antes de rodar o silver_to_gold."
    )

print(f"OK: {len(indicator_cols)} indicadores em StringType (6 estados preservados)")
print(f"Edição {reference_year} | municípios: {silver_df.count()}")

## Step 2 - `dim_municipio` (1 linha por município × edição)

- Origem: colunas cadastrais e descritivas da `silver.munic_clean_data`
- Deriva o `porte_municipio` a partir de `populacao_total` (Seção 3.1): Pequeno I (até 20 mil), Pequeno II (20–50 mil), Médio (50–100 mil), Grande I (100–500 mil) e Grande II (mais de 500 mil)

In [0]:
# porte_municipio derivado da população (Seção 3.1 da especificação)
porte_municipio_expr = (
    F.when(F.col("populacao_total") <= 20000, "Pequeno I")
     .when(F.col("populacao_total") <= 50000, "Pequeno II")
     .when(F.col("populacao_total") <= 100000, "Médio")
     .when(F.col("populacao_total") <= 500000, "Grande I")
     .otherwise("Grande II")
)

dim_municipio = silver_df.select(
    "cod_ibge", "ano_edicao", "nome", "uf_sigla", "regiao",
    "populacao_total", "faixa_populacao",
    porte_municipio_expr.alias("porte_municipio"),
)

upsert_delta_table(
    df_source=dim_municipio,
    table_name=dim_municipio_table,
    merge_keys=["cod_ibge", "ano_edicao"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): dim_municipio com porte populacional derivado",
)
display(dim_municipio.limit(10))

## Step 3 - Radar IMDM: parametrização dos 64 itens pontuados

- Cada item vale no máximo 1 ponto; cada dimensão = pontos obtidos / pontos possíveis × 100
- Formato da configuração: `(dimensao, variável-âncora do item, regra, variáveis-membro)`
- Em itens multi-variável, os pontos ficam apenas na variável-âncora (ex.: `mgov0711`); as demais variáveis preservam o estado original para auditoria (Seção 3.2.1)

**Regras de pontuação:** `SIM_POSITIVO` (Sim = 1), `CATEGORICO_ESCALONADO` (níveis pleno/parcial/ausência), `ESCADA_MAIOR_NIVEL` (maior nível alcançado), `GRUPO_PROPORCIONAL` (fração de Sim do grupo) e `NAO_PONTUA` (fora do índice)

In [0]:
# Radar IMDM (docs/radar_imdm.md): 64 itens, cada um valendo no máximo 1 ponto
# Formato da tupla: (dimensao, variável-âncora do item, regra, variáveis-membro)
SIM_POS    = "SIM_POSITIVO"
CAT_ESC    = "CATEGORICO_ESCALONADO"
ESCADA     = "ESCADA_MAIOR_NIVEL"
GRUPO      = "GRUPO_PROPORCIONAL"
NAO_PONTUA = "NAO_PONTUA"

D1 = "D1_Infra_Acesso"
D2 = "D2_Transparencia"
D3 = "D3_Servicos"
D4 = "D4_Controle_Participacao"
D5 = "D5_LGPD"

imdm_items = [
    # D1 — Infraestrutura, capacidade de TIC e acesso digital (7 itens)
    (D1, "mtic06",  SIM_POS, ["mtic06"]),    # Estrutura organizacional de TI
    (D1, "mtic08",  SIM_POS, ["mtic08"]),    # Centro de Processamento de Dados (CPD)
    (D1, "mtic071", SIM_POS, ["mtic071"]),   # Servidores efetivos próprios na TI
    (D1, "mtic091", SIM_POS, ["mtic091"]),   # Software p/ uso interno (12 meses)
    (D1, "mtic092", SIM_POS, ["mtic092"]),   # Software p/ atendimento à sociedade
    (D1, "mtic19",  SIM_POS, ["mtic19"]),    # Programa/ação de inclusão digital
    (D1, "mtic22",  SIM_POS, ["mtic22"]),    # Wi-Fi público à população
    # D2 — Transparência ativa e dados abertos (11 itens)
    (D2, "mgov01",   SIM_POS, ["mgov01"]),    # Legislação municipal de LAI
    (D2, "mgov061",  SIM_POS, ["mgov061"]),   # Execução orçamentária em tempo real
    (D2, "mgov0711", CAT_ESC, ["mgov0711", "mgov0712", "mgov0713", "mgov0715"]),  # Formato do orçamento
    (D2, "mgov0721", CAT_ESC, ["mgov0721", "mgov0722", "mgov0723", "mgov0725"]),  # Formato das receitas
    (D2, "mgov0731", CAT_ESC, ["mgov0731", "mgov0732", "mgov0733", "mgov0735"]),  # Formato das despesas
    (D2, "mgov0741", CAT_ESC, ["mgov0741", "mgov0742", "mgov0743", "mgov0745"]),  # Formato dos balanços
    (D2, "mgov0751", CAT_ESC, ["mgov0751", "mgov0752", "mgov0753", "mgov0755"]),  # Formato da prestação de contas (LRF)
    (D2, "mgov0761", CAT_ESC, ["mgov0761", "mgov0762", "mgov0763", "mgov0765"]),  # Formato de compras e licitações
    (D2, "mtic272",  SIM_POS, ["mtic272"]),   # Portal de dados abertos (mtic271 fora do índice)
    (D2, "mgov081",  GRUPO,   ["mgov081", "mgov082", "mgov083", "mgov084", "mgov088"]),  # Conteúdo do portal — Grupo A
    (D2, "mgov085",  GRUPO,   ["mgov085", "mgov086", "mgov087", "mgov089"]),            # Conteúdo do portal — Grupo B
    # D3 — Serviços digitais ao cidadão (26 itens)
    (D3, "mtic10", CAT_ESC, ["mtic10"]),     # Situação da página da prefeitura
    *[(D3, f"mtic12b{i}", SIM_POS, [f"mtic12b{i}"]) for i in range(1, 14)],  # mtic12b1..mtic12b13: serviços online
    (D3, "mtic161",  SIM_POS, ["mtic161"]),  # Envio de SMS ao cidadão
    (D3, "mtic162",  SIM_POS, ["mtic162"]),  # Recebimento de SMS do cidadão
    (D3, "mtic163",  SIM_POS, ["mtic163"]),  # Aplicativos da prefeitura/órgãos
    (D3, "mtic164",  SIM_POS, ["mtic164"]),  # Aplicativos de terceiros a partir de dados
    (D3, "mtic165",  SIM_POS, ["mtic165"]),  # Website adaptado a dispositivos móveis
    (D3, "mtic166",  SIM_POS, ["mtic166"]),  # Emissão de boletos/guias de tributos
    (D3, "mtic18a2", SIM_POS, ["mtic18a2"]), # Cadastro unificado p/ todos os serviços
    (D3, "mtic18a3", SIM_POS, ["mtic18a3"]), # Acesso com a conta GOV.BR
    (D3, "mtic171",  ESCADA,  ["mtic171", "mtic172", "mtic173", "mtic174", "mtic175", "mtic176"]),  # Escada: canal de solicitação
    (D3, "mtic014",  SIM_POS, ["mtic014"]),  # Atendimento à distância por WhatsApp
    (D3, "mtic016",  SIM_POS, ["mtic016"]),  # Atendimento por telefone exclusivo
    (D3, "mtic29",   CAT_ESC, ["mtic29"]),   # Modelo de acessibilidade digital
    # D4 — Controle interno e participação (16 itens)
    (D4, "mgov09",  CAT_ESC, ["mgov09"]),    # Estrutura de controle interno (instrumento legal)
    (D4, "mgov17",  SIM_POS, ["mgov17"]),    # Carreira de auditor/controlador
    (D4, "mgov191", SIM_POS, ["mgov191"]),   # Auditoria governamental
    (D4, "mgov192", SIM_POS, ["mgov192"]),   # Controladoria
    (D4, "mgov193", SIM_POS, ["mgov193"]),   # Corregedoria
    (D4, "mgov194", SIM_POS, ["mgov194"]),   # Ouvidoria
    (D4, "mgov195", SIM_POS, ["mgov195"]),   # Integridade / combate à corrupção
    (D4, "mgov201", SIM_POS, ["mgov201"]),   # Controle prévio
    (D4, "mgov202", SIM_POS, ["mgov202"]),   # Controle concomitante
    (D4, "mgov203", SIM_POS, ["mgov203"]),   # Controle posterior
    (D4, "mgov23",  SIM_POS, ["mgov23"]),    # Controle interno na internet
    (D4, "mgov05",  CAT_ESC, ["mgov05"]),    # Publicação de perguntas/respostas LAI (exceção mgov04)
    (D4, "mtic251", SIM_POS, ["mtic251"]),   # Consulta pública online
    (D4, "mtic252", SIM_POS, ["mtic252"]),   # Grupos de discussão online
    (D4, "mtic253", SIM_POS, ["mtic253"]),   # Enquete online
    (D4, "mtic254", SIM_POS, ["mtic254"]),   # Votação online
    # D5 — Proteção de dados pessoais / LGPD (4 itens)
    (D5, "mgov32", SIM_POS, ["mgov32"]),     # Responsável pela LGPD
    (D5, "mgov34", SIM_POS, ["mgov34"]),     # Capacitação em segurança/LGPD (24 meses)
    (D5, "mgov35", SIM_POS, ["mgov35"]),     # Canal sobre uso de dados pessoais
    (D5, "mgov36", SIM_POS, ["mgov36"]),     # Regulamentação municipal de LGPD
]

# Sanidade contra a referência do radar (64 itens; 7/11/26/16/4 por dimensão)
from collections import Counter

assert len(imdm_items) == 64, f"Radar IMDM deve ter 64 itens, encontrados {len(imdm_items)}"
dim_counts = Counter(d for d, _, _, _ in imdm_items)
assert dim_counts == {D1: 7, D2: 11, D3: 26, D4: 16, D5: 4}, f"Distribuição por dimensão divergente: {dim_counts}"

all_members = [v for _, _, _, members in imdm_items for v in members]
assert len(all_members) == len(set(all_members)), "Uma variável pertence a mais de um item do radar"
missing_members = [v for v in all_members if v not in indicator_cols]
assert not missing_members, f"Variáveis do radar ausentes na silver: {missing_members}"

# cod_variable -> (dimensão, âncora do item, regra, é âncora?)
imdm_scoring_map = {
    v: (d, anchor, rule, v == anchor)
    for d, anchor, rule, members in imdm_items
    for v in members
}
print(
    f"Radar OK: 64 itens | {len(imdm_scoring_map)} variáveis pontuadas | "
    f"{len(indicator_cols) - len(imdm_scoring_map)} variáveis NAO_PONTUA"
)

## Step 4 - `dim_indicador` (1 linha por variável × edição; 360 linhas)

- Base do JOIN: as 360 colunas reais da silver via `dq_column_profile` (Ajuste 2) — descarta os códigos fantasmas do dicionário do IBGE e herda os 7 significados reconstruídos na Etapa 2 + a correção de `mtic30`
- `grupo_pergunta`: pai confirmado na `dq_skiplogic_candidates`; caso contrário, radical extraído por regex
- `dimensao_imdm`, `item_imdm`, `peso_indicador` e `regra_pontuacao` vêm do radar do Step 3
- Papel duplo (COBIT APO14.02): glossário de negócios ativo + tabela de parametrização — mudar peso ou regra de um indicador exige apenas atualizar esta tabela

In [0]:
# Base: as 360 colunas reais da silver (via dq_column_profile — Ajuste 2)
profile_df = (
    spark.table(dq_profile_table)
    .filter(F.col("coluna").isin(indicator_cols))
    .select(
        F.col("coluna").alias("cod_variable"),
        F.col("significado"),
        F.col("origem_significado"),
        F.col("tipo_detectado"),
    )
)

# Correção documental do grupo (Ajuste 2, Seção 1.1): o dicionário oficial do IBGE
# traz "educação" em mtic30 por erro de texto; o correto é a área de COMUNICAÇÃO
# (consistente com o bloco mtic31/mtic331 de comunicação social).
CORRECAO_MTIC30 = "Estrutura organizacional para a área de comunicação - existência"
profile_df = profile_df.withColumns(
    {
        "significado": F.when(F.col("cod_variable") == "mtic30", F.lit(CORRECAO_MTIC30)).otherwise(F.col("significado")),
        "origem_significado": F.when(F.col("cod_variable") == "mtic30", F.lit("corrigido pelo grupo")).otherwise(F.col("origem_significado")),
    }
)

# Fallback de significado a partir do dicionário silver
dict_df = (
    spark.table(silver_dict_table)
    .filter(F.col("ano_edicao") == reference_year)
    .select("cod_variable", "meaning")
)

# Pais confirmados por salto lógico (Etapa 2)
skiplogic_df = spark.table(dq_skiplogic_table).select(
    F.col("coluna_filha").alias("cod_variable"),
    F.col("coluna_pai_candidata").alias("pai_confirmado"),
)

# Parametrização do radar (Step 3) como lookup
radar_lookup = spark.createDataFrame(
    [
        (cod, d, anchor, rule, 1.0 if is_anchor else None)
        for cod, (d, anchor, rule, is_anchor) in imdm_scoring_map.items()
    ],
    schema="cod_variable string, dimensao_imdm string, item_imdm string, "
          "regra_pontuacao string, peso_indicador double",
)

dim_indicador = (
    profile_df
    .join(dict_df, on="cod_variable", how="left")
    .join(skiplogic_df, on="cod_variable", how="left")
    .join(radar_lookup, on="cod_variable", how="left")
    .select(
        "cod_variable",
        F.lit(reference_year).cast("int").alias("ano_edicao"),
        F.coalesce("significado", "meaning").alias("meaning"),
        "origem_significado",
        F.when(F.col("cod_variable").startswith("mtic"), "TIC").otherwise("GOV").alias("tema"),
        F.col("tipo_detectado").alias("tipo_resposta"),
        # grupo_pergunta: pai confirmado na dq_skiplogic_candidates; senão radical por regex
        # (Seção 3.2.1: ex. mtic12a1 -> mtic12). Questão-base sem sufixo não tem pai -> NULL.
        F.coalesce(
            F.when(F.col("pai_confirmado") != "", F.col("pai_confirmado")),
            F.when(
                F.regexp_extract("cod_variable", r"^(mtic|mgov)\d{2}", 0) != F.col("cod_variable"),
                F.regexp_extract("cod_variable", r"^(mtic|mgov)\d{2}", 0),
            ),
        ).alias("grupo_pergunta"),
        F.coalesce("dimensao_imdm", F.lit("Nao_Pontua")).alias("dimensao_imdm"),
        F.col("item_imdm"),          # NULL quando Nao_Pontua
        F.col("peso_indicador"),      # 1.0 na variável-âncora do item, NULL nas demais
        F.coalesce("regra_pontuacao", F.lit("NAO_PONTUA")).alias("regra_pontuacao"),
    )
)

upsert_delta_table(
    df_source=dim_indicador,
    table_name=dim_indicador_table,
    merge_keys=["cod_variable", "ano_edicao"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): dim_indicador com regras de pontuação do radar IMDM",
)
display(dim_indicador.limit(10))

## Step 5 - `fact_resposta`: despivotização atômica (5.570 × 360 = 2.005.200 linhas)

- `stack` converte as 360 colunas de indicadores em linhas (município × variável)
- `classe_preenchimento`: as 5 classes mutuamente exclusivas validadas na Etapa 2 — `Valida`, `Nao_Aplicavel_Skip` (salto lógico `-`), `Nao_Sabe`, `Nao_Informou` e `Recusa`
- `resposta_normalizada`: `'1'`/`'0'` para os estados binários; categorias e texto livre preservados; NULL em `Recusa`

In [0]:
# Despivota as 360 colunas de indicadores em linhas atômicas (município × variável)
stack_pairs = ", ".join(f"'{c}', `{c}`" for c in indicator_cols)
resposta_long = silver_df.select(
    "cod_ibge", "ano_edicao",
    F.expr(f"stack({len(indicator_cols)}, {stack_pairs}) as (cod_variable, resposta_original)"),
).withColumn("resposta_original", F.col("resposta_original").cast("string"))

# classe_preenchimento: as 5 classes mutuamente exclusivas validadas na Etapa 2
classe_preenchimento_expr = (
    F.when(F.col("resposta_original") == "Recusa", "Recusa")
     .when(F.col("resposta_original") == "Não sabe informar", "Nao_Sabe")
     .when(F.col("resposta_original") == "Não informou", "Nao_Informou")
     .when(F.col("resposta_original") == "-", "Nao_Aplicavel_Skip")
     .when(F.col("resposta_original").isNull() | (F.trim(F.col("resposta_original")) == ""), "Nao_Informou")
     .otherwise("Valida")
)

# resposta_normalizada: '1'/'0' para estados binários; categoria/texto preservados; NULL em Recusa
resposta_normalizada_expr = (
    F.when(F.col("resposta_original") == "Sim", "1")
     .when(F.col("resposta_original").isin("Não", "-", "Não sabe informar", "Não informou"), "0")
     .when(F.col("resposta_original").isNull() | (F.trim(F.col("resposta_original")) == ""), "0")
     .when(F.col("resposta_original") == "Recusa", F.lit(None).cast("string"))
     .otherwise(F.col("resposta_original"))
)

fact_resposta_base = (
    resposta_long
    .withColumn("classe_preenchimento", classe_preenchimento_expr)
    .withColumn("resposta_normalizada", resposta_normalizada_expr)
    .join(
        dim_indicador.select(
            "cod_variable", "ano_edicao", "dimensao_imdm", "item_imdm",
            "regra_pontuacao", "peso_indicador",
        ),
        on=["cod_variable", "ano_edicao"],
        how="left",
    )
)
print(f"Linhas atômicas: {fact_resposta_base.count()} (esperado: 2.005.200)")

## Step 6 - `fact_resposta`: pontuação no nível do item

Regras que leem várias variáveis do mesmo item — os pontos são atribuídos à variável-âncora:

- **Formato de publicação** (6 itens): aberto = 1,0; só PDF/imagem/outro = 0,5; não publica = 0,0
- **Escada de canais de solicitação** (`mtic171`–`176`): site ou app = 1,0; telefone ou outro website = 0,5; só redes sociais = 0,25; nenhum = 0,0
- **Grupos A e B** (conteúdo do portal): fração de 'Sim' entre as variáveis do grupo
- **Escalonados categóricos** (`mtic10`, `mgov09`, `mtic29`): nível pleno = 1,0; parcial = 0,5; ausência = 0,0
- **Exceção de `mgov05`**: 'Não publica' + `mgov04` = 'Não' (não recebeu pedidos pela LAI) → sai do denominador

Semântica do denominador (Seções 3.3.2 e 3.3.3): `-`, 'Não sabe informar' e 'Não informou' pontuam 0 com peso ativo; `Recusa` anula pontos e peso.

In [0]:
from functools import reduce

item_scores_parts = []

# 1) Formato de publicação (6 itens): aberto = 1.0; só PDF/imagem/outro = 0.5; não publica = 0.0
for prefix in ["071", "072", "073", "074", "075", "076"]:
    x1, x2, x3, x5 = (f"mgov{prefix}{i}" for i in [1, 2, 3, 5])
    is_recusa = F.col(x1) == "Recusa"
    item_scores_parts.append(
        silver_df.select(
            "cod_ibge", "ano_edicao",
            F.lit(x1).alias("cod_variable"),
            F.when(is_recusa, F.lit(None).cast("double"))
             .when(F.col(x1) == "Sim", F.lit(1.0))
             .when((F.col(x2) == "Sim") | (F.col(x3) == "Sim"), F.lit(0.5))
             .otherwise(F.lit(0.0)).alias("pontos_item"),
            F.when(is_recusa, F.lit(None).cast("double"))
             .otherwise(F.lit(1.0)).alias("peso_item"),
        )
    )

# 2) Escada de canais (mtic171–176): site/app = 1.0; telefone/outro website = 0.5; só redes = 0.25; nenhum = 0.0
is_recusa = F.col("mtic171") == "Recusa"
item_scores_parts.append(
    silver_df.select(
        "cod_ibge", "ano_edicao",
        F.lit("mtic171").alias("cod_variable"),
        F.when(is_recusa, F.lit(None).cast("double"))
         .when((F.col("mtic172") == "Sim") | (F.col("mtic174") == "Sim"), F.lit(1.0))
         .when((F.col("mtic171") == "Sim") | (F.col("mtic173") == "Sim"), F.lit(0.5))
         .when(F.col("mtic175") == "Sim", F.lit(0.25))
         .otherwise(F.lit(0.0)).alias("pontos_item"),
        F.when(is_recusa, F.lit(None).cast("double"))
         .otherwise(F.lit(1.0)).alias("peso_item"),
    )
)

# 3) Grupos proporcionais: fração de 'Sim' entre as variáveis do grupo
for anchor, members in [
    ("mgov081", ["mgov081", "mgov082", "mgov083", "mgov084", "mgov088"]),  # Grupo A
    ("mgov085", ["mgov085", "mgov086", "mgov087", "mgov089"]),              # Grupo B
]:
    n = len(members)
    is_recusa = F.col(anchor) == "Recusa"
    sim_count = reduce(
        lambda a, b: a + b,
        [F.when(F.col(c) == "Sim", F.lit(1.0)).otherwise(F.lit(0.0)) for c in members],
    )
    item_scores_parts.append(
        silver_df.select(
            "cod_ibge", "ano_edicao",
            F.lit(anchor).alias("cod_variable"),
            F.when(is_recusa, F.lit(None).cast("double"))
             .otherwise(sim_count / F.lit(float(n))).alias("pontos_item"),
            F.when(is_recusa, F.lit(None).cast("double"))
             .otherwise(F.lit(1.0)).alias("peso_item"),
        )
    )

# 4) Escalonados categóricos simples: nível pleno = 1.0; parcial = 0.5; ausência = 0.0
escalonado_specs = [
    ("mtic10", {"Ativa": 1.0, "Em manutenção ou reestruturação": 0.5}),
    ("mgov09", {"Sim, instituída por instrumento legal": 1.0, "Sim, sem instrumento legal": 0.5}),
    ("mtic29", {
        "Modelo federal de acessibilidade digital (e-MAG)": 1.0,
        "Modelo global de acessibilidade digital (WCAG/WAI)": 1.0,
        "Modelo estadual de acessibilidade digital": 1.0,
        "Modelo próprio de acessibilidade digital": 0.5,
    }),
]
for anchor, value_points in escalonado_specs:
    is_recusa = F.col(anchor) == "Recusa"
    pontos_nivel = F.coalesce(
        *[F.when(F.col(anchor) == v, F.lit(p)) for v, p in value_points.items()]
    )
    item_scores_parts.append(
        silver_df.select(
            "cod_ibge", "ano_edicao",
            F.lit(anchor).alias("cod_variable"),
            F.when(is_recusa, F.lit(None).cast("double"))
             .otherwise(F.coalesce(pontos_nivel, F.lit(0.0))).alias("pontos_item"),
            F.when(is_recusa, F.lit(None).cast("double"))
             .otherwise(F.lit(1.0)).alias("peso_item"),
        )
    )

# 5) mgov05: 'Não publica' + mgov04 = 'Não' (sem pedidos LAI) sai do denominador (pontos e peso NULL)
item_scores_parts.append(
    silver_df.select(
        "cod_ibge", "ano_edicao",
        F.lit("mgov05").alias("cod_variable"),
        F.when(F.col("mgov05") == "Recusa", F.lit(None).cast("double"))
         .when((F.col("mgov05") == "Não publica") & (F.col("mgov04") == "Não"), F.lit(None).cast("double"))
         .when(F.col("mgov05") == "Perguntas e respostas", F.lit(1.0))
         .when(F.col("mgov05") == "Somente as respostas", F.lit(0.5))
         .otherwise(F.lit(0.0)).alias("pontos_item"),
        F.when(F.col("mgov05") == "Recusa", F.lit(None).cast("double"))
         .when((F.col("mgov05") == "Não publica") & (F.col("mgov04") == "Não"), F.lit(None).cast("double"))
         .otherwise(F.lit(1.0)).alias("peso_item"),
    )
)

item_scores = reduce(lambda a, b: a.unionByName(b), item_scores_parts)
print(f"Âncoras pontuadas no nível de item: {item_scores.count()} (esperado: 13)")

## Step 7 - `fact_resposta`: montagem final e gravação

- `pontos_obtidos` e `peso_aplicado` ficam apenas na variável-âncora do item; variáveis `NAO_PONTUA`, membros não-âncora e células de `Recusa` recebem NULL nas duas colunas
- No 'não se aplica' legítimo de `mgov05`, a `resposta_normalizada` e o `pontos_obtidos` ficam NULL (Seção 3.3.3)
- Gravação via `upsert_delta_table` com chave composta `(cod_ibge, cod_variable, ano_edicao)`

In [0]:
# Pontos: SIM_POSITIVO é resolvido na própria linha; regras de item usam os pontos do Step 6
pontos_obtidos_expr = (
    F.when(F.col("regra_pontuacao") == "NAO_PONTUA", F.lit(None).cast("double"))
     .when(F.col("classe_preenchimento") == "Recusa", F.lit(None).cast("double"))
     .when(F.col("peso_indicador").isNull(), F.lit(None).cast("double"))   # membro não-âncora
     .when(
         F.col("regra_pontuacao") == SIM_POS,
         F.when(F.col("resposta_original") == "Sim", F.col("peso_indicador"))   # 1.0 × peso
         .otherwise(F.lit(0.0)),                                                 # 'Não'/'-'/'Não sabe'/'Não informou'
     )
     .otherwise(
         # 'não se aplica' legítimo de mgov05 -> pontos NULL, sai do denominador (Seção 3.3.3)
         F.when(
             (F.col("cod_variable") == "mgov05")
             & (F.col("peso_item").isNull())
             & (F.col("classe_preenchimento") == "Valida"),
             F.lit(None).cast("double"),
         ).otherwise(F.coalesce("pontos_item", F.lit(0.0)))   # demais regras de item (âncora)
     )
)

# Peso: ativo no denominador, exceto NAO_PONTUA, Recusa, não-âncoras e a exceção de mgov05
peso_aplicado_expr = (
    F.when(F.col("regra_pontuacao") == "NAO_PONTUA", F.lit(None).cast("double"))
     .when(F.col("classe_preenchimento") == "Recusa", F.lit(None).cast("double"))
     .when(F.col("peso_indicador").isNull(), F.lit(None).cast("double"))
     .when(F.col("regra_pontuacao") == SIM_POS, F.col("peso_indicador"))
     .otherwise(F.col("peso_item"))
)

fact_resposta = (
    fact_resposta_base
    .join(item_scores, on=["cod_ibge", "ano_edicao", "cod_variable"], how="left")
    .withColumn(
        # 'não se aplica' legítimo de mgov05 -> resposta_normalizada NULL (Seção 3.3.3)
        "resposta_normalizada",
        F.when(
            (F.col("cod_variable") == "mgov05")
            & (F.col("peso_item").isNull())
            & (F.col("classe_preenchimento") == "Valida"),
            F.lit(None).cast("string"),
        ).otherwise(F.col("resposta_normalizada")),
    )
    .select(
        "cod_ibge", "cod_variable", "ano_edicao",
        "resposta_original", "resposta_normalizada", "classe_preenchimento",
        pontos_obtidos_expr.alias("pontos_obtidos"),
        peso_aplicado_expr.alias("peso_aplicado"),
    )
)

upsert_delta_table(
    df_source=fact_resposta,
    table_name=fact_resposta_table,
    merge_keys=["cod_ibge", "cod_variable", "ano_edicao"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): fact_resposta atômica com pontuação por item e denominador efetivo",
)
display(fact_resposta.limit(10))

## Step 8 - `fact_imdm_municipio` (1 linha por município × edição)

- `Score_d = (Σ pontos_obtidos / Σ peso_aplicado) × 100` para cada dimensão (escala 0 a 100)
- `IMDM = média simples das 5 dimensões`; NULL restrito às 7 recusas formais do censo (`motivo_imdm_nulo = 'Recusa'`)
- `nivel_maturidade_mmd`: 4 estágios com corte nos quartis da escala 0–100

In [0]:
dims = {D1: "d1", D2: "d2", D3: "d3", D4: "d4", D5: "d5"}
score_names = {
    "d1": "score_d1_infra_acesso",
    "d2": "score_d2_transparencia",
    "d3": "score_d3_servicos_digitais",
    "d4": "score_d4_controle_participacao",
    "d5": "score_d5_lgpd",
}

# Soma pontos e pesos por dimensão, além da flag de recusa do município
agg_exprs = [
    F.max(F.when(F.col("classe_preenchimento") == "Recusa", 1).otherwise(0)).alias("flg_recusa")
]
for dim_name, k in dims.items():
    agg_exprs.append(
        F.sum(F.when(F.col("dimensao_imdm") == dim_name, F.col("pontos_obtidos"))).alias(f"pts_{k}")
    )
    agg_exprs.append(
        F.sum(F.when(F.col("dimensao_imdm") == dim_name, F.col("peso_aplicado"))).alias(f"peso_{k}")
    )

dimensao_scores = (
    fact_resposta
    .join(
        dim_indicador.select("cod_variable", "ano_edicao", "dimensao_imdm"),
        on=["cod_variable", "ano_edicao"],
        how="left",
    )
    .groupBy("cod_ibge", "ano_edicao")
    .agg(*agg_exprs)
)

# Score da dimensão = (pontos / peso) × 100, protegido contra denominador zero/nulo
score_exprs = [
    F.when(F.col(f"peso_{k}") > 0, F.round(F.col(f"pts_{k}") / F.col(f"peso_{k}") * 100, 2))
     .otherwise(F.lit(None).cast("double")).alias(score_names[k])
    for k in dims.values()
]

fact_imdm_municipio = (
    dimensao_scores
    .select("cod_ibge", "ano_edicao", "flg_recusa", *score_exprs)
    .withColumn(
        # IMDM = média simples das 5 dimensões (NULL quando alguma dimensão não pontua)
        "score_global_imdm",
        F.round(
            (F.col("score_d1_infra_acesso") + F.col("score_d2_transparencia")
             + F.col("score_d3_servicos_digitais") + F.col("score_d4_controle_participacao")
             + F.col("score_d5_lgpd")) / F.lit(5.0),
            2,
        ),
    )
    .withColumn(
        "motivo_imdm_nulo",
        F.when(F.col("flg_recusa") == 1, F.lit("Recusa"))
         .when(F.col("score_global_imdm").isNull(), F.lit("Sem_Item_Valido"))
         .otherwise(F.lit(None).cast("string")),
    )
    .withColumn(
        # Estágios MMD com corte nos quartis da escala 0-100
        "nivel_maturidade_mmd",
        F.when(F.col("score_global_imdm").isNull(), F.lit(None).cast("string"))
         .when(F.col("score_global_imdm") < 25, F.lit("1-Incipiente"))
         .when(F.col("score_global_imdm") < 50, F.lit("2-Basico"))
         .when(F.col("score_global_imdm") < 75, F.lit("3-Intermediario"))
         .otherwise(F.lit("4-Avancado")),
    )
    .select(
        "cod_ibge", "ano_edicao", "motivo_imdm_nulo",
        "score_d1_infra_acesso", "score_d2_transparencia", "score_d3_servicos_digitais",
        "score_d4_controle_participacao", "score_d5_lgpd",
        "score_global_imdm", "nivel_maturidade_mmd",
    )
)

upsert_delta_table(
    df_source=fact_imdm_municipio,
    table_name=fact_imdm_table,
    merge_keys=["cod_ibge", "ano_edicao"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): fact_imdm_municipio com scores D1-D5, IMDM global e estágio MMD",
)
display(fact_imdm_municipio.limit(10))

## Step 9 - Controles automatizados de qualidade (Seção 6)

1. **Volume de indicadores:** 360 variáveis (198 TIC + 162 GOV), 0 sem dimensão/regra e significados preenchidos (7 reconstruídas + `mtic30`)
2. **Conservação exata:** 2.005.200 linhas e a distribuição exata das 6 classes da Etapa 2
3. **Isolamento das recusas:** exatamente 7 IMDM NULL com motivo 'Recusa' e 5.563 scores válidos em [0, 100]
4. **Integridade referencial:** 0 registros órfãos nas FKs fato → dimensões
5. **Proteção ética:** `mgov15`/`mgov16` (dados descritivos de servidores) fora do índice

Qualquer FAIL interrompe o pipeline com `AssertionError`.

In [0]:
dim_municipio_gold  = spark.table(dim_municipio_table).filter(F.col("ano_edicao") == reference_year)
dim_indicador_gold = spark.table(dim_indicador_table).filter(F.col("ano_edicao") == reference_year)
fact_resposta_gold = spark.table(fact_resposta_table).filter(F.col("ano_edicao") == reference_year)
fact_imdm_gold     = spark.table(fact_imdm_table).filter(F.col("ano_edicao") == reference_year)

checks = []
def check(controle, esperado, obtido):
    checks.append((controle, str(esperado), str(obtido), "PASS" if esperado == obtido else "FAIL"))

# 1) Volume de indicadores
tema_counts = {r["tema"]: r["count"] for r in dim_indicador_gold.groupBy("tema").count().collect()}
check("dim_indicador: 360 variáveis", 360, dim_indicador_gold.count())
check("dim_indicador: temas TIC/GOV (198/162)", {"TIC": 198, "GOV": 162}, tema_counts)
check(
    "dim_indicador: 0 variáveis sem dimensão/regra",
    0,
    dim_indicador_gold.where(F.col("dimensao_imdm").isNull() | F.col("regra_pontuacao").isNull()).count(),
)
reconstruidas = ["mtic11", "mtic31", "mtic331", "mtic2421", "mtic2431", "mgov141", "mgov142"]
check(
    "Ajuste 2: significados preenchidos (7 reconstruídas)",
    0,
    dim_indicador_gold.where(F.col("cod_variable").isin(reconstruidas) & F.col("meaning").isNull()).count(),
)
# Correção documental do grupo: mtic30 é COMUNICAÇÃO (o IBGE traz 'educação' por erro)
mtic30_meaning = dim_indicador_gold.where(F.col("cod_variable") == "mtic30").select("meaning").collect()[0]["meaning"]
check(
    "Ajuste 2: mtic30 corrigido para 'comunicação'",
    True,
    (mtic30_meaning is not None) and ("comunicação" in mtic30_meaning.lower()),
)

# 2) Conservação exata de observações e das 6 classes da Etapa 2
classe_dist = {
    r["classe_preenchimento"]: r["count"]
    for r in fact_resposta_gold.groupBy("classe_preenchimento").count().collect()
}
check("fact_resposta: 2.005.200 linhas", 2005200, fact_resposta_gold.count())
check(
    "fact_resposta: distribuição das 6 classes da Etapa 2",
    {"Valida": 1493935, "Nao_Aplicavel_Skip": 500916, "Nao_Informou": 4413,
     "Nao_Sabe": 3416, "Recusa": 2520},
    classe_dist,
)

# 3) Evidência da Etapa 2 e isolamento dos municípios sem IMDM
null_imdm = fact_imdm_gold.where(F.col("score_global_imdm").isNull())
check("fact_imdm: 5.570 municípios", 5570, fact_imdm_gold.count())
check("fact_imdm: exatamente 7 IMDM NULL", 7, null_imdm.count())
check("fact_imdm: motivo 'Recusa' em todos os NULL", 7, null_imdm.where(F.col("motivo_imdm_nulo") == "Recusa").count())
valid_stats = fact_imdm_gold.where(F.col("score_global_imdm").isNotNull()).agg(
    F.count("*").alias("n"), F.min("score_global_imdm").alias("mn"), F.max("score_global_imdm").alias("mx")
).collect()[0]
check("fact_imdm: 5.563 municípios com IMDM válido", 5563, valid_stats["n"])
check("fact_imdm: scores dentro de [0, 100]", True, (valid_stats["mn"] >= 0) and (valid_stats["mx"] <= 100))

# 4) Integridade referencial (FK)
check(
    "FK: fact_resposta -> dim_municipio (0 órfãos)",
    0,
    fact_resposta_gold.join(dim_municipio_gold.select("cod_ibge", "ano_edicao"), ["cod_ibge", "ano_edicao"], "left_anti").count(),
)
check(
    "FK: fact_resposta -> dim_indicador (0 órfãos)",
    0,
    fact_resposta_gold.join(dim_indicador_gold.select("cod_variable", "ano_edicao"), ["cod_variable", "ano_edicao"], "left_anti").count(),
)
check(
    "FK: fact_imdm -> dim_municipio (0 órfãos)",
    0,
    fact_imdm_gold.join(dim_municipio_gold.select("cod_ibge", "ano_edicao"), ["cod_ibge", "ano_edicao"], "left_anti").count(),
)

# 5) Proteção ética/privacidade de servidores
check(
    "Ética: mgov15/mgov16 com regra NAO_PONTUA",
    2,
    dim_indicador_gold.where(
        F.col("cod_variable").isin("mgov15", "mgov16") & (F.col("regra_pontuacao") == "NAO_PONTUA")
    ).count(),
)

# Seção 3.3.3: no 'não se aplica' legítimo de mgov05, pontos_obtidos também é NULL
check(
    "Ajuste 3.3.3: mgov05 não-aplica sem pontos (NULL)",
    0,
    fact_resposta_gold.where(
        (F.col("cod_variable") == "mgov05")
        & (F.col("peso_aplicado").isNull())
        & (F.col("classe_preenchimento") == "Valida")
        & F.col("pontos_obtidos").isNotNull()
    ).count(),
)

# Seção 3.2.1: grupo_pergunta nunca fica com string vazia — pai confirmado, radical regex ou NULL
check(
    "Ajuste 3.2.1: grupo_pergunta sem strings vazias",
    0,
    dim_indicador_gold.where(F.col("grupo_pergunta") == "").count(),
)

results = spark.createDataFrame(checks, schema="controle string, esperado string, obtido string, status string")
display(results)

failed = [c[0] for c in checks if c[3] == "FAIL"]
assert not failed, f"Controles de qualidade da camada Gold REPROVADOS: {failed}"
print("Todos os controles de qualidade da Etapa 3 (Seção 6) passaram.")

In [0]:
%sql
SELECT cod_ibge, pontos_obtidos
From munic_catalog.gold.fact_resposta
WHERE pontos_obtidos IS NOT NULL AND pontos_obtidos <> 1 AND pontos_obtidos <> 0

In [0]:
%sql
SELECT *
From munic_catalog.gold.dim_indicador

In [0]:
%sql
SELECT *
From munic_catalog.gold.fact_imdm_municipio
WHERE 